# Lengkapi Website Master (Otomatis)

Notebook ini mengisi `institution_name`, `domain`, `province_name`, dan `bps_region_code` di `website_master.csv` secara otomatis, dengan cara:
1. Ambil `domain` dari `page_url` di `page_sample_updated.csv` + `excluded_websites.csv`
2. Petakan `domain` -> `province_name` pakai kamus mapping (bisa diedit di Cell 3)
3. `institution_name` dibentuk dari pola "Dinas Sosial Provinsi {province_name}" (edit kalau institusinya beda)
4. Join `province_name` ke tabel kode BPS (`download.xls`) untuk dapat `bps_region_code`

**File yang perlu di-upload di Cell 1:**
- `website_master.csv`
- `page_sample_updated.csv`
- `excluded_websites.csv`
- `download.xls` (tabel Kode Relasi BPS-Kemendagri dari sig.bps.go.id)

## Cell 1 — Upload Semua File

In [ ]:
from google.colab import files

print('Upload: website_master.csv, page_sample_updated.csv, excluded_websites.csv, download.xls')
uploaded = files.upload()

## Cell 2 — Load Semua Data (deteksi nama file otomatis)

In [ ]:
import pandas as pd
import io

def find_file(keyword_list, exclude=None):
    for f in uploaded.keys():
        fl = f.lower()
        if exclude and exclude in fl:
            continue
        if all(k in fl for k in keyword_list):
            return f
    raise FileNotFoundError(f'File dengan keyword {keyword_list} tidak ditemukan di hasil upload')

master_filename = find_file(['website_master']) if any('website_master' in f.lower() for f in uploaded) else find_file(['master'])
page_sample_filename = find_file(['page_sample'])
excluded_filename = find_file(['excluded'])
bps_filename = next(f for f in uploaded.keys() if f.lower().endswith(('.xls', '.xlsx')))

print('website_master  :', master_filename)
print('page_sample     :', page_sample_filename)
print('excluded_websites:', excluded_filename)
print('BPS relation file:', bps_filename)

master = pd.read_csv(io.BytesIO(uploaded[master_filename]))
ps = pd.read_csv(io.BytesIO(uploaded[page_sample_filename]))
ex = pd.read_csv(io.BytesIO(uploaded[excluded_filename]))

bps_html = uploaded[bps_filename].decode('utf-8', errors='ignore')
bps_tables = pd.read_html(io.StringIO(bps_html))
bps = bps_tables[0]
bps.columns = ['no', 'nama_provinsi_bps', 'kode_provinsi_bps', 'nama_provinsi_kemendagri', 'kode_provinsi_kemendagri']
bps = bps.drop(columns='no')

print('\nmaster:', master.shape, '| page_sample:', ps.shape, '| excluded:', ex.shape, '| bps:', bps.shape)

## Cell 3 — Ambil `domain` dari page_url + Mapping `domain` -> `province_name`

Kalau ada `website_id` baru yang belum ada di `province_map`, tambahkan manual di dictionary ini.

In [ ]:
# Gabungkan page_sample + excluded_websites, ambil domain paling sering muncul per website_id
allp = pd.concat([ps, ex], ignore_index=True).dropna(subset=['page_url'])
allp['domain'] = allp['page_url'].str.extract(r'https?://([^/]+)/?')
domain_map = allp.groupby('website_id')['domain'].agg(lambda x: x.value_counts().index[0]).to_dict()

print('Domain yang terdeteksi per website_id:')
for wid, dom in domain_map.items():
    print(f'  {wid}: {dom}')

# Mapping domain -> nama provinsi (edit/tambah kalau ada domain baru yang belum ke-cover)
province_map = {
    'dinsos.acehprov.go.id': 'Aceh',
    'dinsos.sumutprov.go.id': 'Sumatera Utara',
    'dinsos.sumbarprov.go.id': 'Sumatera Barat',
    'dinsos.riau.go.id': 'Riau',
    'dinsosjambipemprov.com': 'Jambi',
    'dinsos.sumselprov.go.id': 'Sumatera Selatan',
    'dinsos.bengkuluprov.go.id': 'Bengkulu',
    'lampungprov.go.id': 'Lampung',
    'drupal.dinsospmd.babelprov.go.id': 'Kepulauan Bangka Belitung',
    'dinsos.kepriprov.go.id': 'Kepulauan Riau',
    'dinsos.jakarta.go.id': 'Dki Jakarta',
    'dinsos.jabarprov.go.id': 'Jawa Barat',
    'jatengprov.go.id': 'Jawa Tengah',
    'dinsos.jogjaprov.go.id': 'Di Yogyakarta',
    'dinsos.jatimprov.go.id': 'Jawa Timur',
    'dinsos.bantenprov.go.id': 'Banten',
    'dissosp3a.baliprov.go.id': 'Bali',
    'dinsosntbpemprov.com': 'Nusa Tenggara Barat',
    'dinsos.nttprov.go.id': 'Nusa Tenggara Timur',
    'dinsos.kalbarprov.go.id': 'Kalimantan Barat',
    'dinsos.kalteng.go.id': 'Kalimantan Tengah',
    'dinsos.kalselprov.go.id': 'Kalimantan Selatan',
    'dinsos.kaltimprov.go.id': 'Kalimantan Timur',
    'dinsos.kaltaraprov.go.id': 'Kalimantan Utara',
    'sulutprov.go.id': 'Sulawesi Utara',
    'sultengprov.go.id': 'Sulawesi Tengah',
    'dinsossulselpemprov.com': 'Sulawesi Selatan',
    'dinsos.sultraprov.go.id': 'Sulawesi Tenggara',
    'dinsos.gorontaloprov.go.id': 'Gorontalo',
    'dinsosp3apmd.sulbarprov.go.id': 'Sulawesi Barat',
    'dinsos.malukuprov.go.id': 'Maluku',
    'malutprov.go.id': 'Maluku Utara',
    'dinsosdukcapil.papua.go.id': 'Papua',
    'dinsos.papuabaratprov.go.id': 'Papua Barat',
}

## Cell 4 — Isi `domain`, `province_name`, `institution_name` ke Master

In [ ]:
master['domain'] = master['website_id'].map(domain_map)
master['province_name'] = master['domain'].map(province_map)

# Ganti pola nama institusi ini kalau bukan Dinas Sosial
master['institution_name'] = master['province_name'].apply(
    lambda p: f'Dinas Sosial Provinsi {p}' if pd.notna(p) else None
)

# Cek website_id yang domainnya gak ketemu / provinsinya gak ke-mapping
missing_domain = master[master['domain'].isna()]
missing_province = master[master['domain'].notna() & master['province_name'].isna()]

if len(missing_domain) > 0:
    print('website_id tanpa domain (tidak ketemu di page_sample/excluded):')
    display(missing_domain[['website_id']])
if len(missing_province) > 0:
    print('domain yang belum ada di province_map, tambahkan manual:')
    display(missing_province[['website_id', 'domain']])

display(master.head(10))

## Cell 5 — Join ke Kode BPS berdasarkan `province_name`

In [ ]:
bps['nama_provinsi_bps_clean'] = bps['nama_provinsi_bps'].str.strip().str.title()
master['province_name_clean'] = master['province_name'].astype(str).str.strip().str.title()

master = master.merge(
    bps[['nama_provinsi_bps_clean', 'kode_provinsi_bps']],
    left_on='province_name_clean', right_on='nama_provinsi_bps_clean',
    how='left'
)
master['bps_region_code'] = master['kode_provinsi_bps']
master = master.drop(columns=['province_name_clean', 'nama_provinsi_bps_clean', 'kode_provinsi_bps'])

missing_code = master[master['province_name'].notna() & master['bps_region_code'].isna()]
if len(missing_code) > 0:
    print('⚠️ Provinsi yang tidak ketemu kode BPS-nya:')
    display(missing_code[['website_id', 'province_name']])
else:
    print('✅ Semua province_name berhasil di-match ke kode BPS.')

display(master)

## Cell 6 — Simpan & Download Hasil

In [ ]:
OUTPUT_FILE = 'website_master_final.csv'
master.to_csv(OUTPUT_FILE, index=False)
print(f'Tersimpan sebagai {OUTPUT_FILE}')

files.download(OUTPUT_FILE)